In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
import astropy.constants as c
import pyneb as pn
import matplotlib.pyplot as plt
HAS_PYNEB=True
from astropy.coordinates import SkyCoord, SkyOffsetFrame
from scipy.stats import linregress
from astropy.visualization import AsinhStretch, PercentileInterval
from skimage.segmentation import find_boundaries
from matplotlib import cm
from skimage.measure import find_contours
from matplotlib.patches import ConnectionPatch
from scipy.optimize import curve_fit
import os
import re
import matplotlib as mpl

plt.rcParams["text.usetex"] = False

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=20)

cmap = cm.get_cmap('magma')


/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_10852/498365531.py:27: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap('magma')


In [ ]:
# load in the HII region catalog
cat_path = 'CATALOGS/flux_catalogs/flux_catalog_ALL_FIELDS.csv'
cat = pd.read_csv(cat_path)
# print all column names
for col in cat.columns:
    print(col)

region_id
npix_region
npix_edge_ring
zoi_mean
zoi_median
ha_F_raw
ha_sigma_F
ha_SNR_raw
ha_b_edge
ha_F_bgsub
ha_SNR_bgsub
hb_F_raw
hb_sigma_F
hb_SNR_raw
hb_b_edge
hb_F_bgsub
hb_SNR_bgsub
oiii5007_F_raw
oiii5007_sigma_F
oiii5007_SNR_raw
oiii5007_b_edge
oiii5007_F_bgsub
oiii5007_SNR_bgsub
sii6716_F_raw
sii6716_sigma_F
sii6716_SNR_raw
sii6716_b_edge
sii6716_F_bgsub
sii6716_SNR_bgsub
sii6731_F_raw
sii6731_sigma_F
sii6731_SNR_raw
sii6731_b_edge
sii6731_F_bgsub
sii6731_SNR_bgsub
nii6584_F_raw
nii6584_sigma_F
nii6584_SNR_raw
nii6584_b_edge
nii6584_F_bgsub
nii6584_SNR_bgsub
oii3727_F_raw
oii3727_sigma_F
oii3727_SNR_raw
oii3727_b_edge
oii3727_F_bgsub
oii3727_SNR_bgsub
field
y
x
removed_by_saddle
removed_by_edge
kind
center_x_px
center_y_px
zoi_center_label
bg_local
sigma_local
radius_p16_px
radius_p50_px
radius_p84_px
radius_p16_pc
radius_p50_pc
radius_p84_pc
radius_areaeq_px
radius_areaeq_pc
boundary_method
area_px_after_carve
radius_areaeq_px_after_carve
radius_areaeq_pc_after_carve
raw_Ha_Hb

In [12]:
# load in the WR and SNR catalogs
wr_catalog = pd.read_csv('CATALOGS/WR_catalog.csv', sep='\t')
snr_catalog = pd.read_csv('CATALOGS/snr_table.csv')
rows = []
with open("CATALOGS/snr_table.csv", "r") as f:
    for line in f:
        # Remove LaTeX and dollar symbols
        clean = re.sub(r"\\rm|\\pm|\$", "", line)
        # Split by comma
        parts = [p.strip() for p in clean.split(",") if p.strip()]
        # Find any token that looks like an RA (contains colons) and a Dec (starts with + or -)
        ra, dec = None, None
        for p in parts:
            if ":" in p and ra is None:
                ra = p
            elif (p.startswith("+") or p.startswith("-")) and ":" in p and dec is None:
                dec = p
        if ra and dec:
            rows.append((ra, dec))

# Make a DataFrame
snr_catalog = pd.DataFrame(rows, columns=["RA", "Dec"])

In [13]:
def field_to_fieldnum(field_label: str) -> str:
    """
    Map your plot field labels to WR catalog Field codes.
    Input examples: 'NE','NW','SE','SW','F5'...'F9'
    Output: 'F1'...'F9'
    """
    mapping = {"NE": "F1", "NW": "F2", "SE": "F3", "SW": "F4"}
    return mapping.get(field_label, field_label)  # F5->F5 etc.


def parse_name_to_coords(name: str) -> SkyCoord:
    # Example: 'J013307.80+302951.1'
    ra_h = name[1:3]
    ra_m = name[3:5]
    ra_s = name[5:10]
    dec_sign = name[10]
    dec_d = name[11:13]
    dec_m = name[13:15]
    dec_s = name[15:]
    ra_str = f"{ra_h}h{ra_m}m{ra_s}s"
    dec_str = f"{dec_sign}{dec_d}d{dec_m}m{dec_s}s"
    return SkyCoord(ra_str, dec_str, frame="icrs")


def load_halpha_log_and_wcs(ha_fits_path: str):
    with fits.open(ha_fits_path) as hdul:
        data = hdul[0].data
        hdr = hdul[0].header

    data = np.where(np.isfinite(data), data, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        ha_log = np.log10(data)
    ha_log = np.where(np.isfinite(ha_log), ha_log, np.nan)

    wcs = WCS(hdr)
    return ha_log, wcs


def try_load_boundary_map(field_label: str, boundary_dir="boundary_map"):
    """
    Tries boundary_map/Boundary_map_{FIELD}.fits where FIELD is e.g. NE, NW, SE, SW, F5...
    Returns array or None if missing/unreadable.
    """
    path = os.path.join(boundary_dir, f"Boundary_map_{field_label}.fits")
    if not os.path.exists(path):
        return None
    try:
        return fits.getdata(path)
    except Exception:
        return None


def get_wr_pixels_for_field(wr_catalog: pd.DataFrame, field_num: str, wcs: WCS):
    wr_field = wr_catalog[wr_catalog["Field"] == field_num]
    if len(wr_field) == 0:
        return np.array([]), np.array([])

    coords = [parse_name_to_coords(n) for n in wr_field["Name (star)"]]
    sky = SkyCoord(coords)
    x_wr, y_wr = wcs.world_to_pixel(sky)
    return np.array(x_wr), np.array(y_wr)


def get_snr_pixels_for_field(snr_catalog: pd.DataFrame, wcs: WCS, xlim=(50, 2000), ylim=(50, 2000)):
    """
    Your snr_table.csv loaded with pd.read_csv.
    This assumes columns named exactly 'RA' and 'Dec' OR 'Dec'/'DEC' variants.
    """
    # Be forgiving about Dec column name
    if "Dec" in snr_catalog.columns:
        dec_col = "Dec"
    elif "DEC" in snr_catalog.columns:
        dec_col = "DEC"
    elif "dec" in snr_catalog.columns:
        dec_col = "dec"
    else:
        raise KeyError("SNR catalog must have a Dec/DEC column (plus RA).")

    if "RA" not in snr_catalog.columns:
        raise KeyError("SNR catalog must have an 'RA' column.")

    coords = SkyCoord(snr_catalog["RA"].astype(str),
                      snr_catalog[dec_col].astype(str),
                      unit=(u.hourangle, u.deg))

    x_snr, y_snr = wcs.world_to_pixel(coords)
    x_snr = np.array(x_snr)
    y_snr = np.array(y_snr)

    in_field = (x_snr > xlim[0]) & (x_snr < xlim[1]) & (y_snr > ylim[0]) & (y_snr < ylim[1])
    return x_snr[in_field], y_snr[in_field]


In [27]:
def _first_existing_path(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


def plot_m33_9panel_bpt_overlay(
    ha_dir,
    ha_filename_pattern,  # e.g. "M33_{FIELD}_Halpha.fits"
    catalog_dir="CATALOGS",
    catalog_pattern="processed_M33_{FIELD}_catalog.csv",
    boundary_dir="boundary_map",
    field_order=("NE", "NW", "F5", "SE", "SW", "F7", "F6", "F8", "F9"),
    xlim=(50, 2000),
    ylim=(50, 2000),
    savepath=None,
):
    """
    3x3 M33 Halpha panels with:
      - global log10(Ha) stretch (1-99 percentile across all fields)
      - optional HII region boundary contours colored by BPT class (skips if catalog or boundary missing)
      - WR stars and SNRs overplotted per field via WCS
      - field label in corner
      - one shared horizontal colorbar

    Missing-data behavior:
      - If boundary map missing -> skip HII contours
      - If processed catalog CSV missing -> skip HII contours
      - If both exist -> plot HII contours
    """

    # --- BPT class colors ---
    # colors_list = [cmap(0.15), "green", cmap(0.7), cmap(0.15)]
    colors_list = ['blue', "green", 'red', 'blue']
    class_colors = {
        "Star-forming": colors_list[0],
        "Composite": colors_list[1],
        "AGN": colors_list[2],
        "Unclassified": colors_list[0],
    }

    # --- Load all Halpha logs + WCS first (for global stretch) ---
    ha_log_by_field = {}
    wcs_by_field = {}
    all_finite = []

    for fld in field_order:
        
        ha_path = os.path.join(ha_dir, ha_filename_pattern.format(FIELD=fld))
        if not os.path.exists(ha_path):
            # try numeric fallback for F5..F9
            if fld.startswith("F"):
                ha_path2 = os.path.join(ha_dir, ha_filename_pattern.format(FIELD=fld[1:]))
                if os.path.exists(ha_path2):
                    ha_path = ha_path2
                else:
                    raise FileNotFoundError(f"Halpha FITS not found for {fld}: tried {ha_path} and {ha_path2}")
            else:
                raise FileNotFoundError(f"Halpha FITS not found for {fld}: tried {ha_path}")

        ha_log, wcs = load_halpha_log_and_wcs(ha_path)
        ha_log_by_field[fld] = ha_log
        wcs_by_field[fld] = wcs

        finite = ha_log[np.isfinite(ha_log)]
        if finite.size:
            all_finite.append(finite)

    if not all_finite:
        raise RuntimeError("No finite Halpha values found across fields.")

    all_finite = np.concatenate(all_finite)
    # vmin, vmax = np.nanpercentile(all_finite, [1, 99])
    vmin, vmax = -18.5, -15

    # --- Figure layout ---
    fig, axes = plt.subplots(3, 3, figsize=(16, 16))
    axes = axes.ravel()
    
    plt.subplots_adjust(
    left=0.02,
    right=0.98,
    bottom=0.0,
    top=1,
    wspace=0.01,
    hspace=0.02,
)

    cmap_gray = cm.get_cmap("gray").copy()
    cmap_gray.set_bad(color="white")

    im_for_cbar = None

    # Legend proxies (single legend for whole figure)
    wr_proxy = plt.Line2D([0], [0], marker="*", linestyle="None",
                          markerfacecolor="lime", markeredgecolor="black",
                          markersize=8, label="WR Stars")
    snr_proxy = plt.Line2D([0], [0], marker="o", linestyle="None",
                           markerfacecolor="orange", markeredgecolor="black",
                           markersize=6, label="SNRs")

    for ax, fld in zip(axes, field_order):
        print(f"Plotting field {fld}...")
        ha_log = ha_log_by_field[fld]
        wcs = wcs_by_field[fld]

        # --- Background Halpha ---
        im = ax.imshow(ha_log, origin="lower", cmap=cmap_gray, vmin=vmin, vmax=vmax, aspect="equal")
        im_for_cbar = im

        # --- Field label ---
        ax.text(
            0.03, 0.97, fld,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=20, color="k",
            bbox=dict(facecolor="magenta", alpha=0.4, edgecolor="none", pad=3),
        )

        # --- Try to locate catalog + boundary paths (both must exist to draw HII contours) ---
        fld_numeric = fld[1:] if fld.startswith("F") else None

        # catalog candidates
        cat_candidates = [
            os.path.join(catalog_dir, catalog_pattern.format(FIELD=fld)),
            os.path.join(catalog_dir, catalog_pattern.format(FIELD=fld_numeric)) if fld_numeric else None,
        ]
        
        cat_path = _first_existing_path(cat_candidates)
        
        # boundary candidates
        bnd_candidates = [
            os.path.join(boundary_dir, f"ContDomain_map_{fld}.fits"),
            os.path.join(boundary_dir, f"ContDomain_map_{fld_numeric}.fits") if fld_numeric else None,
        ]
        print(bnd_candidates)
        bnd_path = _first_existing_path(bnd_candidates)

        # --- HII contours only if BOTH catalog and boundary exist ---
        if (cat_path is not None) and (bnd_path is not None):
            try:
                catalog_processed = pd.read_csv(cat_path)
                bnd_map = fits.getdata(bnd_path)

                bnd_int = np.rint(np.nan_to_num(bnd_map, nan=0)).astype(np.int32)

                for _, row in catalog_processed.iterrows():
                    region_id = row["region_id"]
                    cls = row.get("raw_BPT_class", "Unclassified")
                    color = class_colors.get(cls, "#888888")

                    mask = (bnd_int == region_id)
                    edge = find_boundaries(mask, mode="outer")
                    if np.any(edge):
                        ax.contour(edge, levels=[0.5], colors=[color],
                                   linewidths=1.1, alpha=0.9)
            except Exception as e:
                # If something unexpected happens (malformed CSV, etc.), just skip contours for that panel
                print(f"[WARN] Skipping HII contours for {fld} due to error: {e}")

        # --- WR overlay ---
        field_num = field_to_fieldnum(fld)
        x_wr, y_wr = get_wr_pixels_for_field(wr_catalog, field_num, wcs)
        if x_wr.size:
            ax.scatter(x_wr, y_wr, marker="*", s=50, color="lime",
                       edgecolor="black", linewidth=0.8, zorder=5)

        # --- SNR overlay ---
        x_snr, y_snr = get_snr_pixels_for_field(snr_catalog, wcs, xlim=xlim, ylim=ylim)
        if x_snr.size:
            ax.scatter(x_snr, y_snr, marker="o", s=10, facecolors="orange",
                     zorder=5, edgecolor="black",)

        # --- Cosmetics / zoom ---
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor("white")
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2.0)     # adjust thickness here
            spine.set_edgecolor("black")

    # --- One shared horizontal colorbar ---
    cbar = fig.colorbar(
        im_for_cbar,
        ax=axes.tolist(),
        orientation="horizontal",
        fraction=0.039,
        pad=0.01,
        aspect=30,
    )
    cbar.set_label(r"$\log_{10}$(H$\alpha$ flux)", fontsize=20)
    cbar.outline.set_edgecolor("black")
    cbar.ax.tick_params(color="black", labelcolor="black")
    cbar.outline.set_linewidth(2.0)


    # --- One legend for the whole figure ---
    fig.legend(handles=[wr_proxy, snr_proxy], loc="lower right",
               bbox_to_anchor=(0.98, 0.05),
               frameon=True)

    if savepath:
        fig.savefig(savepath, dpi=300, facecolor="white", bbox_inches="tight")

    plt.show()


In [ ]:
plot_9_panel=True

# IMPORTANT: set these to match your actual Halpha FITS location and naming
ha_dir = "../M33-Maps-Calibrated"
ha_filename_pattern = "M33-{FIELD}/M33{FIELD}-Haflux.fits"  # <-- change to your real pattern

# If your processed catalogs are named exactly like:
# 'CATALOGS/processed_M33_NW_catalog.csv'
# then set:
catalog_dir = "CATALOGS"
catalog_pattern = "flux_catalogs/flux_catalog_{FIELD}.csv"
if plot_9_panel:
    plot_m33_9panel_bpt_overlay(
        ha_dir=ha_dir,
        ha_filename_pattern=ha_filename_pattern,
        catalog_dir=catalog_dir,
        catalog_pattern=catalog_pattern,
        boundary_dir="Boundary_maps/Boundary_map_100pc",
        field_order=("NE", "NW", "F5", "SE", "SW", "F7", "F6", "F8", "F9"),
        xlim=(50, 2000),
        ylim=(50, 2000),
        savepath="plots/M33_allfields_9panel_BPT_overlay.pdf",
    )


/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_10852/3246850790.py:90: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap_gray = cm.get_cmap("gray").copy()


Plotting field NE...
['Boundary_maps/Boundary_map_100pc/ContDomain_map_NE.fits', None]


In [ ]:
def plot_bpt(df,
             x_col='raw_log_NII_Ha_dered',
             y_col='raw_log_OIII_Hb_dered',
             sii_col='raw_sii6731_F_dered',
             sii_e_col='raw_sii6731_sigma_dered',
             class_col='raw_BPT_class',
             nii_col='raw_sii6731_F_dered', ha_col='raw_Ha_F_dered',
             nii_e_col='raw_sii6731_sigma_dered', ha_e_col='raw_Ha_sigma_dered',
             oiii_col='raw_oiii5007_F_dered', hb_col='raw_Hb_F_dered',
             oiii_e_col='raw_oiii5007_sigma_dered', hb_e_col='raw_Ha_sigma_dered',
             snr_cut=3.0b
             figsize=(14, 6),
             savepath=None,
             color_col=None,                 # e.g. 'surface_brightness', 'Rgal', etc.
             cmap='viridis',                 # colormap name or Colormap
             vmin=None, vmax=None,           # color scaling; None => inferred from data
             colorbar=True,
             colorbar_label=None,
             colorbar_log=False,             # True => log-normalize color scale
             missing_color='lightgray'        # used if some points have NaN color_col
            ):
    """
    Plot BPT diagrams:
      LEFT:  [NII]/Ha vs [OIII]/Hb
      RIGHT: [SII]/Ha vs [OIII]/Hb

    Classification is shown via marker shape (and legend).
    If color_col is provided, point facecolor is mapped to that column (with optional colorbar).
    """

    df = df.copy()

    # ---------------------------------------------------
    # Apply SNR cut
    # ---------------------------------------------------
    snr_cols = [f'SNR_{line}' for line in ['Halpha', 'Hbeta', '[NII]6583', '[OIII]5007', '[SII]6731']]
    if all(col in df.columns for col in snr_cols):
        mask = np.ones(len(df), dtype=bool)
        for col in snr_cols:
            mask &= df[col] >= snr_cut
        df = df[mask]
    else:
        print("Warning: SNR columns not found; skipping SNR cut.")

    # ---------------------------------------------------
    # Compute log ratios if missing
    # ---------------------------------------------------
    if x_col not in df.columns:
        df['log_NII_Ha'] = np.log10(df[nii_col] / df[ha_col])

    if 'log_SII_Ha' not in df.columns:
        df['log_SII_Ha'] = np.log10(df[sii_col] / df[ha_col])

    if y_col not in df.columns:
        df['log_OIII_Hb'] = np.log10(df[oiii_col] / df[hb_col])

    # ---------------------------------------------------
    # Error propagation
    # ---------------------------------------------------
    # --- NII/Ha ---
    R_NII_Ha = df[nii_col] / df[ha_col]
    R_NII_Ha_e = R_NII_Ha * np.sqrt(
        (df[nii_e_col] / df[nii_col])**2 +
        (df[ha_e_col] / df[ha_col])**2
    )
    xerr_nii = (1 / np.log(10)) * (R_NII_Ha_e / R_NII_Ha)

    # --- SII/Ha ---
    R_SII_Ha = df[sii_col] / df[ha_col]
    R_SII_Ha_e = R_SII_Ha * np.sqrt(
        (df[sii_e_col] / df[sii_col])**2 +
        (df[ha_e_col] / df[ha_col])**2
    )
    xerr_sii = (1 / np.log(10)) * (R_SII_Ha_e / R_SII_Ha)

    # --- OIII/Hb ---
    R_OIII_Hb = df[oiii_col] / df[hb_col]
    R_OIII_Hb_e = R_OIII_Hb * np.sqrt(
        (df[oiii_e_col] / df[oiii_col])**2 +
        (df[hb_e_col] / df[hb_col])**2
    )
    yerr = (1 / np.log(10)) * (R_OIII_Hb_e / R_OIII_Hb)

    # ---------------------------------------------------
    # Classification styling
    # ---------------------------------------------------
    # (Used when color_col is None; otherwise shapes carry class info and colorbar carries color_col info)
    default_cmap = plt.cm.magma
    class_colors = {
        'Star-forming': default_cmap(0.15),
        'Composite': 'green',
        'AGN': default_cmap(0.7),
        'Unclassified': default_cmap(0.3),
    }

    shapes = {
        'Star-forming': 'o',
        'Composite': '^',
        'AGN': 's',
        'Unclassified': 'x',
    }

    # ---------------------------------------------------
    # Color mapping setup (if requested)
    # ---------------------------------------------------
    use_third_color = (color_col is not None) and (color_col in df.columns)

    norm = None
    sm = None
    if use_third_color:
        cvals_all = df[color_col].to_numpy()
        finite = np.isfinite(cvals_all)

        if vmin is None:
            vmin = np.nanmin(cvals_all[finite]) if np.any(finite) else 0.0
        if vmax is None:
            vmax = np.nanmax(cvals_all[finite]) if np.any(finite) else 1.0

        if colorbar_log:
            # avoid <=0 for LogNorm
            good = cvals_all[finite]
            good = good[good > 0]
            if good.size == 0:
                raise ValueError(f"colorbar_log=True but {color_col} has no positive finite values.")
            if vmin <= 0:
                vmin = np.nanmin(good)
            norm = mpl.colors.LogNorm(vmin=vmin, vmax=vmax)
        else:
            norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

        cmap_obj = mpl.colormaps.get_cmap(cmap) if isinstance(cmap, str) else cmap
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap_obj)

    # ---------------------------------------------------
    # Figure setup
    # ---------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # ---------------------------------------------------
    # LEFT: [NII]/Ha BPT
    # ---------------------------------------------------
    x_vals = np.linspace(-2, 1.0, 400)
    kewley_y = 0.61 / (x_vals - 0.47) + 1.19
    kauff_y  = 0.61 / (x_vals - 0.05) + 1.3

    for cls in shapes.keys():
        subset = df[df[class_col] == cls]
        if len(subset) == 0:
            continue

        # errorbars in gray (no markers)
        ax1.errorbar(subset['log_NII_Ha'], subset['log_OIII_Hb'],
                     xerr=xerr_nii.loc[subset.index],
                     yerr=yerr.loc[subset.index],
                     fmt='none', ecolor='lightgray',
                     elinewidth=0.8, capsize=1.5, zorder=1)

        # markers: either class color OR third-parameter color
        if use_third_color:
            cvals = subset[color_col].to_numpy()
            ok = np.isfinite(cvals)
            # valid colored points
            ax1.scatter(subset.loc[ok, 'log_NII_Ha'], subset.loc[ok, 'log_OIII_Hb'],
                        c=cvals[ok], cmap=sm.cmap, norm=sm.norm,
                        marker=shapes[cls], s=25, alpha=1.0,
                        edgecolors='none', zorder=2, label=cls)
            # missing colored points
            if np.any(~ok):
                ax1.scatter(subset.loc[~ok, 'log_NII_Ha'], subset.loc[~ok, 'log_OIII_Hb'],
                            c=missing_color, marker=shapes[cls], s=25, alpha=1.0,
                            edgecolors='none', zorder=2, label=None)
        else:
            ax1.scatter(subset['log_NII_Ha'], subset['log_OIII_Hb'],
                        c=class_colors.get(cls, 'k'),
                        marker=shapes[cls], s=25, alpha=1.0,
                        edgecolors='none', zorder=2, label=cls)

    ax1.plot(x_vals, kauff_y, 'k--', lw=1.5, label='Kauffmann+03')
    ax1.plot(x_vals, kewley_y, 'k-',  lw=1.5, label='Kewley+01')

    ax1.set_xlim(-1.75, 0)
    ax1.set_ylim(-1.5, 1.5)
    ax1.set_xlabel(r'$\log([\mathrm{N\,II}]/\mathrm{H}\alpha)$')
    ax1.set_ylabel(r'$\log([\mathrm{O\,III}]/\mathrm{H}\beta)$')
    ax1.legend(frameon=False, loc='lower left', fontsize=10)

    # ---------------------------------------------------
    # RIGHT: [SII]/Ha BPT
    # ---------------------------------------------------
    x_vals_sii = np.linspace(-2, 0.5, 400)
    kewley_sii = 0.72 / (x_vals_sii - 0.32) + 1.30

    for cls in shapes.keys():
        subset = df[df[class_col] == cls]
        if len(subset) == 0:
            continue

        ax2.errorbar(subset['log_SII_Ha'], subset['log_OIII_Hb'],
                     xerr=xerr_sii.loc[subset.index],
                     yerr=yerr.loc[subset.index],
                     fmt='none', ecolor='lightgray',
                     elinewidth=0.8, capsize=1.5, zorder=1)

        if use_third_color:
            cvals = subset[color_col].to_numpy()
            ok = np.isfinite(cvals)
            ax2.scatter(subset.loc[ok, 'log_SII_Ha'], subset.loc[ok, 'log_OIII_Hb'],
                        c=cvals[ok], cmap=sm.cmap, norm=sm.norm,
                        marker=shapes[cls], s=25, alpha=1.0,
                        edgecolors='none', zorder=2)
            if np.any(~ok):
                ax2.scatter(subset.loc[~ok, 'log_SII_Ha'], subset.loc[~ok, 'log_OIII_Hb'],
                            c=missing_color, marker=shapes[cls], s=25, alpha=1.0,
                            edgecolors='none', zorder=2)
        else:
            ax2.scatter(subset['log_SII_Ha'], subset['log_OIII_Hb'],
                        c=class_colors.get(cls, 'k'),
                        marker=shapes[cls], s=25, alpha=1.0,
                        edgecolors='none', zorder=2)

    ax2.plot(x_vals_sii, kewley_sii, 'k-', lw=1.5, label='Kewley+01')

    ax2.set_xlim(-1.75, 0)
    ax2.set_ylim(-1.5, 1.5)
    ax2.set_xlabel(r'$\log([\mathrm{S\,II}]/\mathrm{H}\alpha)$')
    ax2.set_ylabel(r'$\log([\mathrm{O\,III}]/\mathrm{H}\beta)$')

    # ticks/spines styling
    ax1.minorticks_on()
    ax2.minorticks_on()
    ax2.tick_params(direction='in', which='both')
    ax1.tick_params(direction='in', which='both')

    for spine in ax1.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)
        spine.set_edgecolor("black")
    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)
        spine.set_edgecolor("black")

    # ---------------------------------------------------
    # Shared colorbar (if requested)
    # ---------------------------------------------------
    if use_third_color and colorbar and sm is not None:
        cax = fig.add_axes([0.99, 0.15, 0.02, 0.7])
        cbar = fig.colorbar(sm, cax=cax)

        if colorbar_label is None:
            colorbar_label = color_col
        cbar.set_label(colorbar_label)

    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches='tight')

    plt.show()
    return fig, (ax1, ax2)
